In [0]:
import dataiku
from dataiku import pandasutils as pdu
import pandas as pd

In [0]:
client = dataiku.api_client()
objects_list = {}

for pk in client.list_project_keys():
    project_handle = client.get_project(pk)
    objects_list[pk] = project_handle.list_recipes(as_type="objects")

In [0]:
for pk, proj_recipes in self.__objects_list.items():
    if records_limit > 0 and records_generated >= records_limit:
        return

    project_handle = self.__client.get_project(pk)

    for r in proj_recipes:
        if records_limit > 0 and records_generated >= records_limit:
            return

        recipe_handle = project_handle.get_recipe(r.id)
        recipe_settings_handle = recipe_handle.get_settings()
        raw_data = recipe_settings_handle.get_recipe_raw_definition()

        next_row = {
            "projectKey": pk,
            "recipe_id": r.id,
            "recipe_type": raw_data["type"],
            "recipe_name": recipe_handle.name,
            "tags": raw_data["tags"],
            "url": self.get_url(r.id, pk),
        }
        try:
            # GUI produces this error message when visiting this recipe's inputs/utputs
            # An invalid argument has been encountered : Failed to iterate, caused by: IllegalArgumentException: No parameters dataset selected for repeating dataset/recipe
            # Seems to happen with the Export To Folder recipe, which exports files to folder.
            # if the user has not set the "Parameters dataset" option for this recipe, or maybe if that dataset has been deleted, then it will throw an exception.

            next_row["engine_parameters"] = raw_data.get("params", {}).get(
                "engineParams", None
            )
            next_row["last_modified_user"] = (
                raw_data.get("versionTag", {})
                .get("lastModifiedBy", {})
                .get("login", None)
            )

            next_row["input_datasets"] = (
                recipe_settings_handle.get_flat_input_refs()
            )
            try:
                next_row["output_datasets"] = (
                    recipe_settings_handle.get_flat_output_refs()
                )
            except Exception as e:
                # this occurs often on Dev-Design.
                print(
                    f"[recipes-generate_rows] [EXPECTED EXCEPTION] Exception in Recipe output dataset, project_key: {pk}, recipe_id: {r.id}: {e}"
                )
        except Exception as e:
            # this occurs often on Dev-Design.
            print(
                f"[recipes-generate_rows] [EXPECTED EXCEPTION] Exception in Recipe input dataset, project_key: {pk}, recipe_id: {r.id}: {e}"
            )
        finally:
            records_generated += 1
            yield next_row
